# Практика 18 · Порівняння архітектур

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ **Цей зошит навчає шістдесят вісім маленьких мереж.** Заміряно: **чотири-пʼять
> хвилин** — 227 с на вільній машині й 305 с на завантаженій, чотири ядра без
> відеокарти, в один потік. Це нормально: предметом теми і є вартість чесного
> порівняння.

Ми не будуватимемо рейтинг архітектур. Ми навчимося ставити порівняння, якому
можна вірити. По ходу зробимо шість замірів:

1. **Розкид від зерна** — скільки дає сама лише зміна випадкового числа. Це
   одиниця виміру для всього далі.
2. **Рецепт проти архітектури** — таблиця 2×2, яка показує, що важить більше.
3. **Ціна порівняння** — скільки прогонів треба, щоб різниця в 0.02 була надійною.
4. **Однаковий бюджет** — три обмеження, три різні переможці.
5. **Девʼять справжніх архітектур** — параметри, час і точність, пораховані самостійно.
6. **Коли архітектура не вирішує** — перенос навчання проти вибору мережі.

In [ ]:
import time
import math

import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

# Один потік: на дрібних тензорах він і швидший за чотири, і — головне —
# детермінований. Під кількома потоками додавання float іде в іншому порядку,
# і числа пливуть від прогону до прогону. Для теми про розкид це неприпустимо.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## Дані: ті самі шість фігур

Датасет той самий, що в темах 13-15: шість класів фігур 28×28, згенерованих
формулами. Нічого не завантажується. Зерно 42 — щоб твої числа збіглися
з лекцією.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=4, noise=0.20):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    """Повертає (count, 1, 28, 28) і (count,). Класи чергуються, тож їх порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        slot = i % len(SHAPE_NAMES)
        images[i, 0] = draw_shape(slot, rng)
        labels[i] = slot
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
train_x, train_y = make_dataset(600, rng)
test_x, test_y = make_dataset(300, rng)

print("навчальна вибірка :", tuple(train_x.shape))
print("перевірочна       :", tuple(test_x.shape))
print("класів            :", len(SHAPE_NAMES),
      "→ вгадування навмання дає", round(1 / len(SHAPE_NAMES), 3))

## Мережі й навчання

Дві архітектури з [теми 14](../14-resnet/lecture.html): блок із двох згорток 3×3
з батчнормом, і той самий блок зі скіп-зʼєднанням. Різниця між ними — рівно один
рядок `out = out + x`, а кількість ваг однакова до одиниці. Це важливо: інакше
ми порівнювали б не архітектури, а розміри.

In [ ]:
class Block(nn.Module):
    """Дві згортки 3×3 з батчнормом. residual=True вмикає скіп-зʼєднання."""

    def __init__(self, channels, residual):
        super().__init__()
        # bias=False, бо батчнорм одразу за згорткою однаково відніме будь-який зсув
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.residual = residual

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + x          # ← оце і є вся різниця між двома архітектурами
        return self.relu(out)


class Net(nn.Module):
    """Стем → n_blocks однакових блоків → глобальне усереднення → лінійний шар."""

    def __init__(self, n_blocks, residual, channels=16, n_classes=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, channels, 3, stride=2, padding=1, bias=False),  # 28×28 → 14×14
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.MaxPool2d(2),                                             # 14×14 → 7×7
        )
        self.blocks = nn.Sequential(*[Block(channels, residual) for _ in range(n_blocks)])
        self.pool = nn.AdaptiveAvgPool2d(1)     # глобальне усереднення з теми 08
        self.fc = nn.Linear(channels, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def accuracy(model, x, y, batch=300):
    """Частка правильних відповідей. Обовʼязково в режимі eval() — див. тему 11."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(x), batch):
            correct += (model(x[i:i + batch]).argmax(1) == y[i:i + batch]).sum().item()
    model.train()
    return correct / len(x)


def train_model(model, x, y, epochs=12, lr=0.03, batch=64, seed=0, time_budget=None):
    """Навчає модель. Повертає (скільки епох зроблено, скільки секунд пішло).

    time_budget задає навчання «на час»: епохи йдуть, доки не вичерпано секунди.
    """
    torch.manual_seed(seed)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_function = nn.CrossEntropyLoss()
    # окремий генератор для перемішування, щоб порядок батчів не залежав від ваг
    order_generator = torch.Generator().manual_seed(seed)
    model.train()
    started = time.perf_counter()
    epochs_done = 0
    while True:
        if time_budget is None and epochs_done >= epochs:
            break
        if time_budget is not None and time.perf_counter() - started >= time_budget:
            break
        order = torch.randperm(len(x), generator=order_generator)
        for start in range(0, len(x), batch):
            chosen = order[start:start + batch]
            optimizer.zero_grad()
            loss_function(model(x[chosen]), y[chosen]).backward()
            optimizer.step()
        epochs_done += 1
    return epochs_done, time.perf_counter() - started


def run_once(n_blocks, residual, channels, recipe, seed, time_budget=None):
    """Один повний прогін: побудувати, навчити, заміряти точність на перевірці."""
    torch.manual_seed(seed)
    model = Net(n_blocks, residual, channels)
    epochs_done, seconds = train_model(model, train_x, train_y, seed=seed,
                                       time_budget=time_budget, **recipe)
    return accuracy(model, test_x, test_y), epochs_done, seconds


# два рецепти, які будуть із нами до кінця зошита
CAREFUL = dict(lr=0.003, epochs=8)     # обережний: маленький крок, мало епох
BOLD = dict(lr=0.03, epochs=20)        # сміливий: крок у десять разів більший

print("проста мережа   :", count_params(Net(3, False, 16)), "ваг")
print("залишкова мережа:", count_params(Net(3, True, 16)), "ваг")
assert count_params(Net(3, False, 16)) == count_params(Net(3, True, 16)), \
    "мережі мають різну кількість ваг — порівняння було б нечесним!"
print("✅ ваг порівну — різниця лише в скіп-зʼєднанні")

## Замір 1 · Розкид від зерна

Найважливіший замір теми, і його треба зробити **першим**. Ми навчаємо одну й ту
саму мережу пʼять разів, змінюючи **лише** зерно: ваги стартують з інших випадкових
чисел, приклади йдуть в іншому порядку. Архітектура, дані й рецепт незмінні.

Робимо це двічі — двома рецептами, бо розкид сам залежить від рецепта.

In [ ]:
SEEDS_SPREAD = [0, 1, 2, 3, 4]
seed_runs = {}       # рецепт → список точностей
seed_time = {}       # рецепт → середній час одного прогону

for recipe_name, recipe in [("обережний", CAREFUL), ("сміливий", BOLD)]:
    accuracies, seconds_list = [], []
    for seed in SEEDS_SPREAD:
        test_accuracy, _, seconds = run_once(3, True, 16, recipe, seed)
        accuracies.append(test_accuracy)
        seconds_list.append(seconds)
    seed_runs[recipe_name] = accuracies
    seed_time[recipe_name] = float(np.mean(seconds_list))

print("рецепт     | пʼять прогонів                        | найгірший | найкращий | розкид | σ")
for recipe_name in ("обережний", "сміливий"):
    values = seed_runs[recipe_name]
    spread = max(values) - min(values)
    sigma = float(np.std(values, ddof=1))
    print(f"{recipe_name:10s} | " + " ".join(f"{v:.3f}" for v in values)
          + f" | {min(values):9.3f} | {max(values):9.3f} | {spread:6.3f} | {sigma:.4f}")

print()
print("Розкид обережного рецепта більший за розкид сміливого у",
      round((max(seed_runs["обережний"]) - min(seed_runs["обережний"]))
            / (max(seed_runs["сміливий"]) - min(seed_runs["сміливий"])), 1), "раза")
print("Середній час одного прогону: обережний %.1f с, сміливий %.1f с"
      % (seed_time["обережний"], seed_time["сміливий"]))

### Перевіримо, що σ — це саме те, що ми думаємо

`np.std(..., ddof=1)` рахує стандартне відхилення вибірки. Порахуємо його руками
за означенням і звіримо: корінь із середнього квадрата відхилення від середнього,
поділеного не на `n`, а на `n − 1`.

In [ ]:
values = np.array(seed_runs["обережний"])
mean_value = values.mean()
# ddof=1 означає ділення на (n − 1): так поправляють оцінку, коли середнє
# ми теж узяли з тих самих даних, а не знали наперед
by_hand = math.sqrt(((values - mean_value) ** 2).sum() / (len(values) - 1))
by_numpy = float(np.std(values, ddof=1))

print("середнє        :", round(float(mean_value), 4))
print("σ руками       :", round(by_hand, 6))
print("σ бібліотекою  :", round(by_numpy, 6))
assert np.allclose(by_hand, by_numpy), "розрахунок розійшовся!"
print("✅ збігається")

## Замір 2 · Рецепт проти архітектури

Головний замір теми. Дві архітектури × два рецепти = чотири клітинки, кожна —
середнє з трьох зерен.

Клітинки із залишковою мережею ми вже порахували в замірі 1 (це та сама
конфігурація), тому їх беремо звідти й не навчаємо вдруге. Довчити треба лише
просту мережу.

In [ ]:
SEEDS_GRID = [0, 1, 2]
grid = {}

# залишкова мережа — беремо перші три зерна із заміру 1, це та сама конфігурація
for recipe_name in ("обережний", "сміливий"):
    grid[("залишкова", recipe_name)] = seed_runs[recipe_name][:len(SEEDS_GRID)]

# просту мережу треба навчити
for recipe_name, recipe in [("обережний", CAREFUL), ("сміливий", BOLD)]:
    grid[("проста", recipe_name)] = [run_once(3, False, 16, recipe, seed)[0]
                                     for seed in SEEDS_GRID]

means = {key: float(np.mean(value)) for key, value in grid.items()}

print("                 | рецепт обережний | рецепт сміливий | різниця від рецепта")
for architecture in ("проста", "залишкова"):
    careful = means[(architecture, "обережний")]
    bold = means[(architecture, "сміливий")]
    print(f"{architecture:16s} | {careful:16.3f} | {bold:15.3f} | {abs(bold - careful):19.3f}")

print("-" * 78)
arch_gap = {name: abs(means[("проста", name)] - means[("залишкова", name)])
            for name in ("обережний", "сміливий")}
print(f"{'різниця від архітектури':16s} | {arch_gap['обережний']:16.3f} "
      f"| {arch_gap['сміливий']:15.3f} |")

recipe_gap = {name: abs(means[(name, "сміливий")] - means[(name, "обережний")])
              for name in ("проста", "залишкова")}
print()
print("НАЙБІЛЬША різниця від рецепта     :", round(max(recipe_gap.values()), 3))
print("НАЙБІЛЬША різниця від архітектури :", round(max(arch_gap.values()), 3))
print("рецепт переважив архітектуру у",
      round(max(recipe_gap.values()) / max(arch_gap.values()), 1), "раза")

### А тепер прикладемо мірило

Різниця між архітектурами існує тільки тоді, коли вона більша за розкид від зерна.
Порівняємо кожну різницю з розкидом того рецепта, під яким її отримано.

In [ ]:
print("рецепт     | різниця архітектур | розкид від зерна | висновок")
for recipe_name in ("обережний", "сміливий"):
    spread = max(seed_runs[recipe_name]) - min(seed_runs[recipe_name])
    difference = arch_gap[recipe_name]
    verdict = ("різницю видно" if difference > spread
               else "різниці НЕ виявлено — вона всередині розкиду")
    print(f"{recipe_name:10s} | {difference:18.3f} | {spread:16.3f} | {verdict}")

## Замір 3 · Скільки коштує чесне порівняння

Скільки прогонів потрібно, щоб різниця розміром Δ була надійною? Виведення —
у розділі 7 лекції, ось підсумкова формула:

```
n ≥ 15.7 × σ² / Δ²
```

Головне в ній — **Δ у квадраті внизу**: захотів побачити вдвічі меншу різницю —
плати вчетверо більше прогонів.

In [ ]:
def runs_needed(sigma, delta):
    """Скільки прогонів на кожну мережу потрібно, щоб різниця delta була надійною.

    15.7 = 2 × (1.96 + 0.84)²: стандартний запас на 5 % хибних тривог
    і 80 % шансів побачити справжню різницю.
    """
    return 15.7 * sigma * sigma / (delta * delta)


print("рецепт     |     σ | Δ=0.02 | Δ=0.05 | Δ=0.10   (прогонів на кожну мережу)")
for recipe_name in ("обережний", "сміливий"):
    sigma = float(np.std(seed_runs[recipe_name], ddof=1))
    row = [math.ceil(runs_needed(sigma, delta)) for delta in (0.02, 0.05, 0.10)]
    print(f"{recipe_name:10s} | {sigma:.4f} | {row[0]:6d} | {row[1]:6d} | {row[2]:6d}")

print()
print("Скільки часу коштує виявити різницю 0.02 між ДВОМА мережами:")
for recipe_name in ("обережний", "сміливий"):
    sigma = float(np.std(seed_runs[recipe_name], ddof=1))
    n = math.ceil(runs_needed(sigma, 0.02))
    total_seconds = 2 * n * seed_time[recipe_name]
    print(f"   {recipe_name:10s}: {n:5d} прогонів на мережу, {2 * n:5d} разом, "
          f"{total_seconds / 60:7.1f} хв")

## Замір 4 · Однаковий бюджет чого?

Три маленькі мережі й три різні способи зрівняти умови. Переможець має мінятися.

* **однакові параметри** — ширину підбираємо так, щоб ваг було приблизно порівну;
* **однаковий час** — природний розмір, кожній рівно три секунди на навчання;
* **однакові епохи** — природний розмір, кожній дванадцять епох.

⚠️ Рядок «однаковий час» — **єдиний у зошиті, який не відтворюється точно**. Скільки
епох устигне мережа за три секунди, залежить від того, наскільки завантажена твоя
машина, тож числа цього рядка в тебе будуть інші, ніж у лекції. Стійке тут інше:
дешевша мережа встигає більше епох, і цей виграш переважує різницю в будові.

In [ ]:
FAMILIES = [("широка", 1, False), ("глибока", 5, False), ("залишкова", 4, True)]
PARAM_TARGET = 14000

# 1. підбираємо ширину під однакову кількість ваг
equal_width = {}
for name, n_blocks, residual in FAMILIES:
    best_channels, best_distance = None, float("inf")
    for channels in range(6, 49):
        distance = abs(count_params(Net(n_blocks, residual, channels)) - PARAM_TARGET)
        if distance < best_distance:
            best_channels, best_distance = channels, distance
    equal_width[name] = best_channels

print("підбір ширини під ≈14 000 ваг:")
for name, n_blocks, residual in FAMILIES:
    channels = equal_width[name]
    print(f"   {name:10s} каналів {channels:2d} → {count_params(Net(n_blocks, residual, channels)):6d} ваг")

In [ ]:
SEEDS_BUDGET = [0, 1, 2]
budget_results = {}

CONSTRAINTS = [
    ("однакові параметри", "params"),
    ("однаковий час", "time"),
    ("однакові епохи", "epochs"),
]

for label, kind in CONSTRAINTS:
    print(f"--- обмеження: {label}")
    for name, n_blocks, residual in FAMILIES:
        channels = equal_width[name] if kind == "params" else 16
        time_budget = 3.0 if kind == "time" else None
        accuracies, epochs_list = [], []
        for seed in SEEDS_BUDGET:
            test_accuracy, epochs_done, _ = run_once(
                n_blocks, residual, channels, dict(lr=0.03, epochs=12), seed,
                time_budget=time_budget)
            accuracies.append(test_accuracy)
            epochs_list.append(epochs_done)
        budget_results[(label, name)] = accuracies
        print(f"   {name:10s} каналів {channels:2d} · "
              f"{count_params(Net(n_blocks, residual, channels)):6d} ваг · "
              f"епох {int(np.mean(epochs_list)):3d} · "
              f"точність {np.mean(accuracies):.3f} "
              f"(від {min(accuracies):.3f} до {max(accuracies):.3f})")

In [ ]:
def spread_of(label, name):
    """Розкид між зернами для однієї клітинки таблиці."""
    values = budget_results[(label, name)]
    return max(values) - min(values)


print("обмеження           |  широка | глибока | залишкова | попереду | відрив | мірило")
for label, _ in CONSTRAINTS:
    row = {name: float(np.mean(budget_results[(label, name)])) for name, _, _ in FAMILIES}
    ordered = sorted(row.items(), key=lambda pair: -pair[1])
    gap = ordered[0][1] - ordered[1][1]
    # мірило — розкид від зерна в тих двох мережах, що змагаються за перше місце
    ruler = max(spread_of(label, ordered[0][0]), spread_of(label, ordered[1][0]))
    print(f"{label:19s} | {row['широка']:7.3f} | {row['глибока']:7.3f} "
          f"| {row['залишкова']:9.3f} | {ordered[0][0]:9s} | {gap:6.3f} | {ruler:6.3f}")

print()
print("Чи можна назвати переможця:")
for label, _ in CONSTRAINTS:
    row = {name: float(np.mean(budget_results[(label, name)])) for name, _, _ in FAMILIES}
    ordered = sorted(row.items(), key=lambda pair: -pair[1])
    gap = ordered[0][1] - ordered[1][1]
    ruler = max(spread_of(label, ordered[0][0]), spread_of(label, ordered[1][0]))
    if gap > ruler:
        print(f"   {label:19s}: так, «{ordered[0][0]}» — відрив {gap:.3f} більший за розкид {ruler:.3f}")
    else:
        print(f"   {label:19s}: НІ — попереду «{ordered[0][0]}», але відрив {gap:.3f} "
              f"менший за розкид {ruler:.3f}")

print()
print("Зверни увагу: попереду щоразу ІНША мережа — обмеження вирішує відповідь.")
print("⚠️ Рядок «однаковий час» залежить від завантаженості машини: у лекції там")
print("   інші числа, бо прогін був іншим. Решта рядків відтворюється точно.")

## Замір 5 · Девʼять справжніх архітектур

`torchvision` уміє зібрати архітектуру **без ваг** — `weights=None`. Нічого не
завантажується, а числа виходять точні. Порахуємо самі: скільки ваг, як вони
поділені між згортковими й лінійними шарами, скільки триває один прохід.

Точність — **чужий замір**: вона лежить в описі ваг у бібліотеці, і ми беремо її
звідти, не завантажуючи самих ваг.

In [ ]:
ARCHITECTURES = [
    ("AlexNet", "alexnet", 2012, 224),
    ("VGG16", "vgg16", 2014, 224),
    ("GoogLeNet", "googlenet", 2014, 224),
    ("ResNet18", "resnet18", 2015, 224),
    ("ResNet50", "resnet50", 2015, 224),
    ("Inception-v3", "inception_v3", 2015, 299),
    ("MobileNet-v2", "mobilenet_v2", 2018, 224),
    ("EfficientNet-B0", "efficientnet_b0", 2019, 224),
    ("MobileNet-v3-small", "mobilenet_v3_small", 2019, 224),
]


def measure_forward_ms(model, sample, repeats=2):
    """Час одного прямого проходу — НАЙКРАЩИЙ із repeats.

    Мінімум, а не середнє: чуже навантаження може лише сповільнити наш прогін,
    тож найшвидший результат найближчий до правди про мережу.
    """
    model.eval()
    with torch.no_grad():
        model(sample)                      # розігрів
        times = []
        for _ in range(repeats):
            started = time.perf_counter()
            model(sample)
            times.append((time.perf_counter() - started) * 1000)
    return min(times)


ARCH_TABLE = []
for title, key, year, side in ARCHITECTURES:
    model = models.get_model(key, weights=None)
    total = sum(p.numel() for p in model.parameters())
    in_conv = sum(p.numel() for module in model.modules()
                  if isinstance(module, nn.Conv2d) for p in module.parameters())
    in_linear = sum(p.numel() for module in model.modules()
                    if isinstance(module, nn.Linear) for p in module.parameters())
    n_conv = sum(1 for module in model.modules() if isinstance(module, nn.Conv2d))
    milliseconds = measure_forward_ms(model, torch.randn(1, 3, side, side))
    # точність із опису ваг — самі ваги при цьому не завантажуються
    published = models.get_model_weights(key).DEFAULT.meta["_metrics"]["ImageNet-1K"]["acc@1"]
    ARCH_TABLE.append(dict(title=title, year=year, params=total,
                           conv=100 * in_conv / total, linear=100 * in_linear / total,
                           n_conv=n_conv, ms=milliseconds, top1=published))

print("архітектура        | рік  |  параметрів | згортки | лінійні | шарів | час, мс | top-1")
for row in ARCH_TABLE:
    print("%-18s | %4d | %11s | %6.1f%% | %6.1f%% | %5d | %7.1f | %5.1f"
          % (row["title"], row["year"], format(row["params"], ",").replace(",", " "),
             row["conv"], row["linear"], row["n_conv"], row["ms"], row["top1"]))

### Звірка: наш підрахунок = підрахунок бібліотеки

`torchvision` зберігає кількість параметрів у метаданих ваг. Наш власний підрахунок
має збігтися з ним до одиниці — це перевірка, що ми рахуємо те саме, що й автори.

Два винятки, і вони повчальні. GoogLeNet та Inception-v3 мають **допоміжні
класифікатори** — додаткові голови, які працюють лише під час навчання й потрібні
були, щоб градієнт доходив до перших шарів. Тому в кожної з цих двох мереж є
**два правильні числа**: з допоміжними головами й без них.

І ось що цікаво: сама бібліотека відповідає на це питання по-різному. Для GoogLeNet
у метаданих записана мережа **без** допоміжних голів, для Inception-v3 — **з ними**.
Це не помилка нашого підрахунку, а нагадування, наскільки крихкі числа в чужих
таблицях: навіть «кількість параметрів» — величина, яка залежить від того, що саме
рахувати.

In [ ]:
# ці дві мережі мають допоміжні голови, які працюють лише під час навчання
WITH_AUX = {"googlenet", "inception_v3"}

for title, key, _, _ in ARCHITECTURES:
    ours = sum(p.numel() for p in models.get_model(key, weights=None).parameters())
    theirs = models.get_model_weights(key).DEFAULT.meta["num_params"]
    if key in WITH_AUX:
        # та сама мережа в робочому вигляді — без допоміжних голів
        working = models.get_model(key, weights=None, aux_logits=False, init_weights=False)
        ours_working = sum(p.numel() for p in working.parameters())
        assert theirs in (ours, ours_working), \
            f"{title}: у нас {ours} або {ours_working}, у бібліотеці {theirs}"
        which = "з допоміжними" if theirs == ours else "без допоміжних"
        print(f"{title:20s} {ours:>11,d} з допоміжними / {ours_working:>11,d} без них; "
              f"бібліотека дає {theirs:>11,d} — {which}".replace(",", " "))
    else:
        assert ours == theirs, f"{title}: у нас {ours}, у бібліотеці {theirs}"
        print(f"{title:20s} {ours:>11,d} == {theirs:>11,d}".replace(",", " "))

print()
print("✅ усі девʼять збіглися з бібліотекою")
print("⚠️ але для GoogLeNet бібліотека рахує мережу БЕЗ допоміжних голів,")
print("   а для Inception-v3 — З ними. Одна бібліотека, два різні означення.")

### Одна архітектура, два рецепти

Найпряміший доказ головної думки теми лежить у самій бібліотеці. Для кількох
архітектур `torchvision` тримає **два набори ваг**: старий і новий. Архітектура
однакова до останнього шару, різниця тільки в рецепті навчання.

In [ ]:
TWO_RECIPES = ["resnet50", "resnet101", "resnext50_32x4d", "wide_resnet50_2",
               "regnet_y_400mf", "mobilenet_v3_large", "efficientnet_b1"]

print("архітектура          | старий рецепт | новий рецепт | приріст")
recipe_gains = []
for key in TWO_RECIPES:
    scores = {}
    for entry in list(models.get_model_weights(key)):
        top1 = entry.meta.get("_metrics", {}).get("ImageNet-1K", {}).get("acc@1")
        if top1 and entry.name.endswith(("V1", "V2")):
            scores[entry.name[-2:]] = top1
    gain = scores["V2"] - scores["V1"]
    recipe_gains.append(gain)
    print(f"{key:20s} | {scores['V1']:13.2f} | {scores['V2']:12.2f} | {gain:+7.2f}")

# порівняємо це з тим, що дає вдвічі глибша мережа за ТИМ САМИМ старим рецептом
resnet50_v1 = models.ResNet50_Weights.IMAGENET1K_V1.meta["_metrics"]["ImageNet-1K"]["acc@1"]
resnet50_v2 = models.ResNet50_Weights.IMAGENET1K_V2.meta["_metrics"]["ImageNet-1K"]["acc@1"]
resnet101_v1 = models.ResNet101_Weights.IMAGENET1K_V1.meta["_metrics"]["ImageNet-1K"]["acc@1"]

print()
print("ResNet50 → ResNet101 за старим рецептом (вдвічі глибше): %+.2f"
      % (resnet101_v1 - resnet50_v1))
print("ResNet50, той самий, новий рецепт                      : %+.2f"
      % (resnet50_v2 - resnet50_v1))
print("ResNet50 з новим рецептом проти ResNet101 зі старим     : %+.2f"
      % (resnet50_v2 - resnet101_v1))

## Замір 6 · Коли архітектура не вирішує

Останній замір. Беремо ті самі три мережі, вчимо їх з нуля на різній кількості
розмічених прикладів — і поруч ставимо четвертий варіант: залишкову мережу
з **чужим тілом**, навченим на трьох інших класах, і замороженими вагами тіла
(перенос навчання з [теми 12](../12-transfer-learning/lecture.html)).

Питання: коли різниця між архітектурами взагалі щось означає?

In [ ]:
# джерельна задача: перші три класи, багато прикладів
source_mask = train_y < 3
source_x, source_y = train_x[source_mask], train_y[source_mask]

torch.manual_seed(0)
source_model = Net(4, True, 16, n_classes=3)
train_model(source_model, source_x, source_y, epochs=25, lr=0.03, seed=0)
print("джерельна модель: %d прикладів трьох класів, точність на них %.3f"
      % (len(source_x), accuracy(source_model, source_x, source_y)))

# тіло забираємо, голову лишаємо позаду: у цілі шість класів, а не три
source_body = {name: value.clone() for name, value in source_model.state_dict().items()
               if not name.startswith("fc.")}
print("перенесемо", len(source_body), "тензорів тіла; голову навчатимемо заново")

In [ ]:
DATA_SIZES = [36, 180, 600]
SEEDS_DATA = [0, 1]
TRANSFER_EPOCHS = 15

transfer_table = {}
for size in DATA_SIZES:
    small_x, small_y = train_x[:size], train_y[:size]
    for name, n_blocks, residual in FAMILIES:
        accuracies = []
        for seed in SEEDS_DATA:
            torch.manual_seed(seed)
            model = Net(n_blocks, residual, 16)
            train_model(model, small_x, small_y, epochs=TRANSFER_EPOCHS, lr=0.03, seed=seed)
            accuracies.append(accuracy(model, test_x, test_y))
        transfer_table[(size, name)] = accuracies

    accuracies = []
    for seed in SEEDS_DATA:
        torch.manual_seed(seed)
        model = Net(4, True, 16)
        model.load_state_dict(source_body, strict=False)
        # тіло заморожене: вчиться лише голова на шість класів
        for parameter in list(model.stem.parameters()) + list(model.blocks.parameters()):
            parameter.requires_grad = False
        train_model(model, small_x, small_y, epochs=TRANSFER_EPOCHS, lr=0.03, seed=seed)
        accuracies.append(accuracy(model, test_x, test_y))
    transfer_table[(size, "перенос")] = accuracies

print("прикладів | широка | глибока | залишкова | перенос | розкид між архітектурами | виграш переносу")
for size in DATA_SIZES:
    scratch = {name: float(np.mean(transfer_table[(size, name)]))
               for name, _, _ in FAMILIES}
    transferred = float(np.mean(transfer_table[(size, "перенос")]))
    spread_between = max(scratch.values()) - min(scratch.values())
    advantage = transferred - max(scratch.values())
    print("%9d | %6.3f | %7.3f | %9.3f | %7.3f | %24.3f | %+15.3f"
          % (size, scratch["широка"], scratch["глибока"], scratch["залишкова"],
             transferred, spread_between, advantage))

## Скільки все це коштувало

In [ ]:
total_seconds = time.perf_counter() - notebook_started
print("зошит виконувався %.0f с (%.1f хв) в один потік без відеокарти"
      % (total_seconds, total_seconds / 60))
print()
print("І це — шість замірів на мережах у 14 тисяч ваг і датасеті з 600 картинок.")
print("Саме тому чесних порівнянь архітектур так мало.")

## Завдання

### 🟢 Рівень 1

Додай до заміру 1 третій рецепт — власний (наприклад, `lr=0.01`, 12 епох) —
і пʼятьма зернами поміряй його розкид. Де він опиниться між 0.330 і 0.020?

### 🟡 Рівень 2

У замірі 4 обмеження «однаковий час» дало три секунди кожній мережі. Пройди
бюджети 1, 2, 4 і 8 секунд і побудуй таблицю «бюджет → переможець». Чи є бюджет,
на якому перемагає глибока мережа?

### 🔴 Рівень 3

Постав чесне порівняння простої й залишкової мережі: візьми сміливий рецепт,
порахуй за формулою потрібну кількість прогонів для Δ = 0.02, зроби рівно стільки
прогонів кожної мережі й напиши висновок. Відповідь «різниці не виявлено» —
повноцінний результат, і саме він найімовірніший.